# 🏛️ US Treasury Bond Issuance Tracker

Interactive dashboard for US Treasury auction data sourced live from
[TreasuryDirect.gov](https://www.treasurydirect.gov).

| | |
|---|---|
| **Coverage** | Bills · Notes · Bonds · TIPS · FRNs · CMBs |
| **History** | 2001 → today |
| **Data source** | TreasuryDirect TA_WS API (public, no key required) |

### How to use
1. **Run all cells** (`Cell → Run All` or `Shift+Enter` through each)
2. Click **🔄 Refresh Data** to fetch the latest data from TreasuryDirect  
   *(first load may take 1–3 minutes for the full history)*
3. Explore each section with its interactive chart / table
4. Click **💾 Export to CSV** (Section 8) to save data locally


In [ ]:
# Run this cell once if any packages are missing, then restart the kernel
# !pip install requests pandas plotly ipywidgets


In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os
from datetime import datetime, date
import warnings
warnings.filterwarnings("ignore")

print("✅ All packages loaded successfully")


In [ ]:
# ─── Configuration ───────────────────────────────────────────────────────────
BASE_URL       = "https://www.treasurydirect.gov/TA_WS/securities"
START_DATE     = "2001-01-01"
END_DATE       = date.today().strftime("%Y-%m-%d")
PAGE_SIZE      = 250          # maximum page size supported by the API

SECURITY_TYPES = ["Bill", "Note", "Bond", "CMB", "TIPS", "FRN"]

TYPE_COLORS = {
    "Bill": "#1f77b4",
    "Note": "#ff7f0e",
    "Bond": "#2ca02c",
    "CMB":  "#d62728",
    "TIPS": "#9467bd",
    "FRN":  "#8c564b",
}

TYPE_COLORS_LIGHT = {
    "Bill": "#dbeafe",
    "Note": "#fef9c3",
    "Bond": "#dcfce7",
    "CMB":  "#fee2e2",
    "TIPS": "#f3e8ff",
    "FRN":  "#ffe4e6",
}

# Columns to coerce to numeric
NUMERIC_COLS = [
    "offeringAmount", "totalTendered", "totalAccepted",
    "competitiveAccepted", "noncompetitiveTendersAccepted", "noncompetitiveAccepted",
    "directBidderAccepted", "directBidderTendersAccepted",
    "indirectBidderAccepted", "indirectBidderTendersAccepted",
    "primaryDealerAccepted", "primaryDealerTendersAccepted",
    "somaAccepted", "somaHoldings", "somaTendersAccepted",
    "fimaNoncompetitiveAccepted", "fimaNoncompetitiveTendersAccepted",
    "treasuryDirectAccepted", "treasuryDirectTendersAccepted",
    "bidToCoverRatio",
    "highYield", "lowYield", "averageMedianYield",
    "highDiscountRate", "averageMedianDiscountRate",
    "highInvestmentRate", "lowInvestmentRate", "averageMedianInvestmentRate",
    "highDiscountMargin", "averageMedianDiscountMargin",
    "highPrice", "lowPrice", "averageMedianPrice", "pricePer100",
    "interestRate", "spread", "frnIndexDeterminationRate",
    "accruedInterestPer1000", "adjustedAccruedInterestPer1000",
    "standardInterestPaymentPer1000",
    "unadjustedAccruedInterestPer1000", "unadjustedPrice", "adjustedPrice",
    "indexRatioOnIssueDate", "refCpiOnIssueDate", "refCpiOnDatedDate",
    "estimatedAmountOfPubliclyHeldMaturingSecuritiesByType",
    "allotmentPercentageOnCompetitiveTenders",
    "nlpExclusionAmount", "nlpReportingThreshold",
    "maximumCompetitiveAward", "maximumNoncompetitiveAward", "maximumSingleBid",
    "minimumBidAmount", "minimumStripAmount", "minimumToIssue",
    "multiplesToBid", "multiplesToIssue",
    "allocatedThroughNoncompetitive",
    "currentlyOutstanding",
]

DATE_COLS = [
    "auctionDate", "issueDate", "maturityDate", "datedDate", "originalDatedDate",
    "announcementDate", "firstInterestPaymentDate", "callDate", "calledDate",
    "frnIndexDeterminationDate", "originalIssueDate", "backDatedDate", "maturingDate",
]

print(f"📅 Data range : {START_DATE} → {END_DATE}")
print(f"🏷️  Security types: {', '.join(SECURITY_TYPES)}")


In [ ]:
# ─── API Fetching Functions ───────────────────────────────────────────────────

def fetch_security_type(sec_type: str, start: str, end: str) -> list:
    """
    Fetch all auction records for one security type using paginated requests.
    Returns a flat list of JSON dicts.
    """
    records = []
    page = 1
    while True:
        url = (
            f"{BASE_URL}/search"
            f"?type={sec_type}"
            f"&startDate={start}"
            f"&endDate={end}"
            f"&pagesize={PAGE_SIZE}"
            f"&pagenum={page}"
            f"&format=json"
        )
        try:
            resp = requests.get(url, timeout=45)
            resp.raise_for_status()
            data = resp.json()
            if not data:
                break
            records.extend(data)
            if len(data) < PAGE_SIZE:
                break          # reached the last page
            page += 1
        except requests.exceptions.HTTPError as e:
            print(f"  ⚠️  HTTP error for {sec_type} page {page}: {e}")
            break
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️  Network error for {sec_type} page {page}: {e}")
            break
        except ValueError:
            print(f"  ⚠️  JSON parse error for {sec_type} page {page}")
            break
    return records


def fetch_upcoming() -> list:
    """Fetch the upcoming auction schedule."""
    url = f"{BASE_URL}/upcoming?format=json"
    try:
        resp = requests.get(url, timeout=45)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"  ⚠️  Upcoming auctions: {e}")
        return []


def clean_dataframe(records: list) -> pd.DataFrame:
    """Convert raw JSON records to a typed, sorted DataFrame."""
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)

    # Parse dates
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Parse numerics
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "auctionDate" in df.columns:
        df = df.sort_values("auctionDate").reset_index(drop=True)
    return df


def load_all_data(start: str = START_DATE, end: str = END_DATE) -> pd.DataFrame:
    """
    Fetch and combine auction data for all six security types.
    Prints progress as each type is fetched.
    """
    all_records = []
    for sec_type in SECURITY_TYPES:
        print(f"  📥 Fetching {sec_type}s … ", end="", flush=True)
        recs = fetch_security_type(sec_type, start, end)
        print(f"{len(recs):,} records")
        all_records.extend(recs)
    return clean_dataframe(all_records)


print("✅ API functions ready")


In [ ]:
# ─── Data Loading & Refresh Widget ───────────────────────────────────────────

# Shared state accessible by all sections
state = {
    "df":       pd.DataFrame(),
    "upcoming": pd.DataFrame(),
}

status_out = widgets.Output()

btn_refresh = widgets.Button(
    description="🔄 Refresh Data",
    button_style="primary",
    layout=widgets.Layout(width="180px", height="38px"),
    tooltip="Fetch latest auction data from TreasuryDirect (takes 1–3 min for full history)",
)

def on_refresh(_btn):
    with status_out:
        clear_output(wait=True)
        print(f"🔄 Connecting to TreasuryDirect …")
        print(f"   Fetching all security types from {START_DATE} to {END_DATE}.")
        print(f"   First load of the full history can take 1–3 minutes — please wait.\n")
        df = load_all_data(START_DATE, END_DATE)
        upcoming_raw = fetch_upcoming()
        state["df"]       = df
        state["upcoming"] = clean_dataframe(upcoming_raw)
        if not df.empty:
            print(f"\n✅ Loaded {len(df):,} auction records")
            print(f"   Types     : {', '.join(sorted(df['securityType'].unique()))}")
            print(f"   Date range: {df['auctionDate'].min().date()} → {df['auctionDate'].max().date()}")
            if "offeringAmount" in df.columns:
                total = df["offeringAmount"].sum()
                print(f"   Total offered: ${total/1e12:.2f} Trillion")
        else:
            print("⚠️  No data returned — check your internet connection.")
        print(f"   Upcoming auctions: {len(upcoming_raw)} scheduled")

btn_refresh.on_click(on_refresh)

display(widgets.VBox([
    widgets.HTML("<h3>▶ Step 1 — Load Data</h3>"
                 "<p>Click the button to fetch data. Subsequent refreshes update all charts.</p>"),
    btn_refresh,
    status_out,
]))


In [ ]:
# ─── Section 1 · Overview ────────────────────────────────────────────────────

def show_overview():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return

    # Auction counts by type
    by_type = df.groupby("securityType").size().reset_index(name="count")

    # Offering amounts (billions)
    amt = pd.DataFrame()
    if "offeringAmount" in df.columns:
        amt = (
            df.groupby("securityType")["offeringAmount"]
            .sum()
            .reset_index()
        )
        amt["bn"] = amt["offeringAmount"] / 1e9

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            "Number of Auctions by Security Type",
            "Total Offering Amount by Type (USD Billions)",
        ],
    )

    fig.add_trace(
        go.Bar(
            x=by_type["securityType"], y=by_type["count"],
            marker_color=[TYPE_COLORS.get(t, "#999") for t in by_type["securityType"]],
            hovertemplate="%{x}: %{y:,} auctions<extra></extra>",
            name="Auctions",
        ),
        row=1, col=1,
    )

    if not amt.empty:
        fig.add_trace(
            go.Bar(
                x=amt["securityType"], y=amt["bn"],
                marker_color=[TYPE_COLORS.get(t, "#999") for t in amt["securityType"]],
                hovertemplate="%{x}: $%{y:,.1f}B<extra></extra>",
                name="Offered",
            ),
            row=1, col=2,
        )

    fig.update_layout(
        title_text="📊 Treasury Issuance — All-Time Overview",
        showlegend=False, height=460, template="plotly_white",
    )
    fig.show()

    # Quick stats
    print(f"\n{'═'*65}")
    print(f"  Records loaded    : {len(df):,}")
    print(f"  Date range        : {df['auctionDate'].min().date()} → {df['auctionDate'].max().date()}")
    if "offeringAmount" in df.columns:
        print(f"  Total offered     : ${df['offeringAmount'].sum()/1e12:.2f} Trillion")
    print(f"  Security types    : {', '.join(sorted(df['securityType'].unique()))}")
    print(f"{'═'*65}")


_out1 = widgets.Output()
_btn1 = widgets.Button(description="📊 Show Overview", button_style="info",
                       layout=widgets.Layout(width="180px"))
_btn1.on_click(lambda _: (_out1.clear_output(wait=True), _out1.__enter__(),
                           show_overview(), _out1.__exit__(None,None,None)))

def _click1(_):
    with _out1:
        clear_output(wait=True)
        show_overview()
_btn1.on_click(_click1)

display(widgets.VBox([widgets.HTML("<h3>Section 1 — Overview</h3>"), _btn1, _out1]))


In [ ]:
# ─── Section 2 · Issuance Amounts Over Time ──────────────────────────────────

def show_issuance_over_time():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return
    if "offeringAmount" not in df.columns or "auctionDate" not in df.columns:
        print("⚠️  Required columns missing."); return

    df2 = df.dropna(subset=["auctionDate","offeringAmount","securityType"]).copy()
    df2["quarter"] = df2["auctionDate"].dt.to_period("Q").dt.to_timestamp()

    grouped = (
        df2.groupby(["quarter","securityType"])["offeringAmount"]
        .sum().reset_index()
    )
    grouped["bn"] = grouped["offeringAmount"] / 1e9

    fig = px.bar(
        grouped, x="quarter", y="bn", color="securityType",
        color_discrete_map=TYPE_COLORS, barmode="stack",
        labels={"quarter":"Quarter","bn":"Offering Amount (USD Billions)","securityType":"Type"},
        title="📈 Total Offering Amount by Quarter (2001–Present)",
        hover_data={"bn":":.1f","securityType":True},
    )
    fig.update_layout(
        height=520, template="plotly_white",
        legend_title="Security Type", hovermode="x unified",
        xaxis_title="Quarter", yaxis_title="Total Offered (USD Billions)",
    )
    fig.show()


_out2 = widgets.Output()
_btn2 = widgets.Button(description="📈 Issuance Over Time", button_style="info",
                       layout=widgets.Layout(width="200px"))
def _click2(_):
    with _out2: clear_output(wait=True); show_issuance_over_time()
_btn2.on_click(_click2)
display(widgets.VBox([widgets.HTML("<h3>Section 2 — Issuance Amounts Over Time</h3>"), _btn2, _out2]))


In [ ]:
# ─── Section 3 · Bid-to-Cover Ratio ─────────────────────────────────────────

def show_bid_to_cover():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return
    if "bidToCoverRatio" not in df.columns:
        print("⚠️  bidToCoverRatio column not available."); return

    df2 = df.dropna(subset=["auctionDate","bidToCoverRatio","securityType"]).copy()
    df2 = df2[df2["bidToCoverRatio"] > 0]
    if df2.empty:
        print("ℹ️  No bid-to-cover data available."); return

    type_opts = sorted(df2["securityType"].unique().tolist())
    dd_type = widgets.SelectMultiple(
        options=type_opts, value=type_opts,
        description="Types:", layout=widgets.Layout(height="150px", width="160px"),
    )
    out_inner = widgets.Output()

    def _update(change=None):
        selected = list(dd_type.value)
        with out_inner:
            clear_output(wait=True)
            df3 = df2[df2["securityType"].isin(selected)].copy()
            extra = []
            for c in ["securityTerm", "offeringAmount", "highYield"]:
                if c in df3.columns: extra.append(c)

            fig = px.scatter(
                df3, x="auctionDate", y="bidToCoverRatio",
                color="securityType", color_discrete_map=TYPE_COLORS,
                opacity=0.55,
                trendline="lowess", trendline_options=dict(frac=0.07),
                labels={"auctionDate":"Auction Date","bidToCoverRatio":"Bid-to-Cover Ratio","securityType":"Type"},
                title="📉 Bid-to-Cover Ratio Over Time",
                hover_data=extra if extra else None,
            )
            fig.update_layout(height=500, template="plotly_white", hovermode="closest")
            fig.show()

    dd_type.observe(_update, names="value")
    _update()
    display(widgets.HBox([
        widgets.VBox([widgets.HTML("<b>Filter types:</b>"), dd_type]),
        out_inner,
    ]))


_out3 = widgets.Output()
_btn3 = widgets.Button(description="📉 Bid-to-Cover", button_style="info",
                       layout=widgets.Layout(width="170px"))
def _click3(_):
    with _out3: clear_output(wait=True); show_bid_to_cover()
_btn3.on_click(_click3)
display(widgets.VBox([widgets.HTML("<h3>Section 3 — Bid-to-Cover Ratio Analysis</h3>"), _btn3, _out3]))


In [ ]:
# ─── Section 4 · Yield Analysis ─────────────────────────────────────────────

def show_yield_analysis():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return

    df2 = df.copy()
    # Best available yield column per row (waterfall)
    df2["yield_pct"] = (
        df2.get("highYield",          pd.Series(dtype=float))
        .fillna(df2.get("highInvestmentRate", pd.Series(dtype=float)))
        .fillna(df2.get("highDiscountRate",   pd.Series(dtype=float)))
        .fillna(df2.get("interestRate",       pd.Series(dtype=float)))
    )
    df2 = df2.dropna(subset=["auctionDate","yield_pct","securityType"])
    df2 = df2[df2["yield_pct"] > 0].sort_values("auctionDate")

    type_opts = sorted(df2["securityType"].unique().tolist())
    dd_type = widgets.SelectMultiple(
        options=type_opts, value=type_opts,
        description="Types:", layout=widgets.Layout(height="150px", width="160px"),
    )
    out_inner = widgets.Output()

    def _update(change=None):
        selected = list(dd_type.value)
        with out_inner:
            clear_output(wait=True)
            df3 = df2[df2["securityType"].isin(selected)].copy()
            extra = [c for c in ["securityTerm","offeringAmount"] if c in df3.columns]
            fig = px.line(
                df3, x="auctionDate", y="yield_pct", color="securityType",
                color_discrete_map=TYPE_COLORS,
                labels={"auctionDate":"Auction Date","yield_pct":"Yield (%)","securityType":"Type"},
                title="💹 Auction Yield Over Time (high yield / investment rate / discount rate)",
                hover_data=extra if extra else None,
            )
            fig.update_layout(height=500, template="plotly_white", hovermode="x unified")
            fig.show()

    dd_type.observe(_update, names="value")
    _update()
    display(widgets.HBox([
        widgets.VBox([widgets.HTML("<b>Filter types:</b>"), dd_type]),
        out_inner,
    ]))


_out4 = widgets.Output()
_btn4 = widgets.Button(description="💹 Yield Analysis", button_style="info",
                       layout=widgets.Layout(width="170px"))
def _click4(_):
    with _out4: clear_output(wait=True); show_yield_analysis()
_btn4.on_click(_click4)
display(widgets.VBox([widgets.HTML("<h3>Section 4 — Yield Analysis</h3>"), _btn4, _out4]))


In [ ]:
# ─── Section 5 · Auction Count & Frequency ───────────────────────────────────

def show_auction_count():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return

    df2 = df.dropna(subset=["auctionDate","securityType"]).copy()
    df2["year"] = df2["auctionDate"].dt.year

    yearly = df2.groupby(["year","securityType"]).size().reset_index(name="count")

    fig = px.bar(
        yearly, x="year", y="count", color="securityType",
        color_discrete_map=TYPE_COLORS, barmode="stack",
        labels={"year":"Year","count":"Number of Auctions","securityType":"Type"},
        title="🗓️ Number of Auctions per Year by Security Type",
    )
    fig.update_layout(
        height=500, template="plotly_white",
        hovermode="x unified", xaxis_title="Year", yaxis_title="Auctions",
    )
    fig.show()


_out5 = widgets.Output()
_btn5 = widgets.Button(description="🗓️ Auction Count", button_style="info",
                       layout=widgets.Layout(width="170px"))
def _click5(_):
    with _out5: clear_output(wait=True); show_auction_count()
_btn5.on_click(_click5)
display(widgets.VBox([widgets.HTML("<h3>Section 5 — Auction Count & Frequency</h3>"), _btn5, _out5]))


In [ ]:
# ─── Section 6 · Upcoming Auction Schedule ───────────────────────────────────

def show_upcoming():
    df_up = state["upcoming"]
    if df_up is None or (isinstance(df_up, pd.DataFrame) and df_up.empty):
        print("ℹ️  No upcoming auctions data — click 🔄 Refresh Data first."); return

    if not isinstance(df_up, pd.DataFrame):
        df_up = pd.DataFrame(df_up)
    if df_up.empty:
        print("ℹ️  No upcoming auctions scheduled."); return

    # Select display columns in priority order
    DISPLAY = [
        "announcementDate","auctionDate","issueDate","maturityDate",
        "securityType","securityTerm","offeringAmount",
        "reopening","callable","strippable","series","cusip","originalCusip",
    ]
    cols = [c for c in DISPLAY if c in df_up.columns]
    df_show = df_up[cols].copy()

    # Format dates
    for col in ["announcementDate","auctionDate","issueDate","maturityDate"]:
        if col in df_show.columns:
            df_show[col] = pd.to_datetime(df_show[col], errors="coerce").dt.strftime("%Y-%m-%d")

    # Format offering amount
    if "offeringAmount" in df_show.columns:
        df_show["offeringAmount"] = pd.to_numeric(df_show["offeringAmount"], errors="coerce")
        df_show["offeringAmount"] = df_show["offeringAmount"].apply(
            lambda x: f"${x/1e9:.2f}B" if pd.notna(x) and x > 0 else "TBA"
        )

    # Sort by auction date
    if "auctionDate" in df_show.columns:
        df_show = df_show.sort_values("auctionDate")

    def _row_bg(row):
        t = row.get("securityType", "")
        c = TYPE_COLORS_LIGHT.get(t, "#f9f9f9")
        return [f"background-color: {c}" for _ in row]

    styled = (
        df_show.style
        .apply(_row_bg, axis=1)
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color","#1e3a5f"),("color","white"),
                ("font-weight","bold"),("padding","6px 14px"),
            ]},
            {"selector": "td", "props": [
                ("border","1px solid #e0e0e0"),("padding","5px 14px"),("font-size","13px"),
            ]},
        ])
    )

    display(HTML(f"<h4>📅 {len(df_show)} upcoming auction(s) scheduled</h4>"))
    display(styled)

    # Bar chart of upcoming offering amounts
    df_chart = df_up.copy()
    if "offeringAmount" in df_chart.columns and "auctionDate" in df_chart.columns:
        df_chart["offeringAmount"] = pd.to_numeric(df_chart["offeringAmount"], errors="coerce")
        df_chart = df_chart.dropna(subset=["auctionDate","offeringAmount","securityType"])
        df_chart = df_chart[df_chart["offeringAmount"] > 0]
        df_chart["auctionDate"] = pd.to_datetime(df_chart["auctionDate"], errors="coerce")
        df_chart["bn"] = df_chart["offeringAmount"] / 1e9
        df_chart["label"] = (
            df_chart["auctionDate"].dt.strftime("%Y-%m-%d") + "<br>" + df_chart["securityType"]
        )
        if not df_chart.empty:
            fig = px.bar(
                df_chart.sort_values("auctionDate"),
                x="auctionDate", y="bn", color="securityType",
                color_discrete_map=TYPE_COLORS, text="bn",
                labels={"auctionDate":"Auction Date","bn":"Offering (USD Billions)","securityType":"Type"},
                title="📅 Upcoming Auction Offering Amounts",
            )
            fig.update_traces(texttemplate="$%{text:.1f}B", textposition="outside")
            fig.update_layout(height=420, template="plotly_white", showlegend=True)
            fig.show()


_out6 = widgets.Output()
_btn6 = widgets.Button(description="📅 Upcoming Schedule", button_style="warning",
                       layout=widgets.Layout(width="190px"))
def _click6(_):
    with _out6: clear_output(wait=True); show_upcoming()
_btn6.on_click(_click6)
display(widgets.VBox([widgets.HTML("<h3>Section 6 — Upcoming Auction Schedule</h3>"), _btn6, _out6]))


In [ ]:
# ─── Section 7 · Recent Auctions — Full Metrics Table ────────────────────────

def show_recent_table():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return

    # Controls
    dd_type = widgets.Dropdown(
        options=["All"] + sorted(df["securityType"].unique().tolist()),
        value="All", description="Type:", layout=widgets.Layout(width="200px"),
    )
    sl_rows = widgets.IntSlider(
        value=50, min=10, max=500, step=10,
        description="Show rows:", layout=widgets.Layout(width="380px"),
        style={"description_width": "90px"},
    )
    out_tbl = widgets.Output()

    # Priority columns to display
    PRIORITY_COLS = [
        "auctionDate","issueDate","maturityDate",
        "securityType","securityTerm",
        "offeringAmount","totalTendered","totalAccepted",
        "bidToCoverRatio",
        "highYield","lowYield","averageMedianYield",
        "highInvestmentRate","averageMedianInvestmentRate",
        "highDiscountRate","averageMedianDiscountRate",
        "highDiscountMargin","averageMedianDiscountMargin",
        "highPrice","lowPrice","averageMedianPrice","pricePer100",
        "interestRate","spread","frnIndexDeterminationRate",
        "competitiveAccepted","noncompetitiveAccepted",
        "directBidderAccepted","indirectBidderAccepted","primaryDealerAccepted",
        "somaAccepted","somaHoldings",
        "fimaNoncompetitiveAccepted","treasuryDirectAccepted",
        "allotmentPercentageOnCompetitiveTenders",
        "refCpiOnIssueDate","indexRatioOnIssueDate",
        "estimatedAmountOfPubliclyHeldMaturingSecuritiesByType",
        "reopening","callable","strippable","floatingRate","backDated",
        "cusip","originalCusip","series","auctionFormat",
        "interestPaymentFrequency","firstInterestPeriod",
    ]

    AMOUNT_COLS = [
        "offeringAmount","totalTendered","totalAccepted",
        "competitiveAccepted","noncompetitiveAccepted",
        "directBidderAccepted","indirectBidderAccepted","primaryDealerAccepted",
        "somaAccepted","somaHoldings","fimaNoncompetitiveAccepted",
        "treasuryDirectAccepted",
        "estimatedAmountOfPubliclyHeldMaturingSecuritiesByType",
    ]

    def _update(change=None):
        with out_tbl:
            clear_output(wait=True)
            df2 = df.copy()
            if dd_type.value != "All":
                df2 = df2[df2["securityType"] == dd_type.value]
            df2 = df2.sort_values("auctionDate", ascending=False).head(sl_rows.value)

            cols = [c for c in PRIORITY_COLS if c in df2.columns]
            df_show = df2[cols].copy()

            # Format dates
            for col in ["auctionDate","issueDate","maturityDate"]:
                if col in df_show.columns:
                    df_show[col] = df_show[col].dt.strftime("%Y-%m-%d")

            # Format $ amounts as billions
            for col in AMOUNT_COLS:
                if col in df_show.columns:
                    df_show[col] = df_show[col].apply(
                        lambda x: f"${x/1e9:.3f}B" if pd.notna(x) and x > 0 else ""
                    )

            styled = df_show.style.set_table_styles([
                {"selector":"th","props":[
                    ("background-color","#1e3a5f"),("color","white"),
                    ("font-size","11px"),("padding","4px 10px"),("white-space","nowrap"),
                ]},
                {"selector":"td","props":[
                    ("font-size","11px"),("padding","3px 10px"),
                    ("border","1px solid #eeeeee"),("white-space","nowrap"),
                ]},
                {"selector":"tr:nth-child(even)","props":[("background-color","#f7f7f7")]},
            ])
            display(HTML(f"<p><b>Showing {len(df_show)} most recent auctions "
                         f"({dd_type.value})</b> — scroll right for all metrics</p>"))
            display(styled)

    dd_type.observe(_update, names="value")
    sl_rows.observe(_update, names="value")
    _update()
    display(widgets.VBox([
        widgets.HBox([dd_type, sl_rows]),
        out_tbl,
    ]))


_out7 = widgets.Output()
_btn7 = widgets.Button(description="🔍 Recent Auctions Table", button_style="info",
                       layout=widgets.Layout(width="220px"))
def _click7(_):
    with _out7: clear_output(wait=True); show_recent_table()
_btn7.on_click(_click7)
display(widgets.VBox([
    widgets.HTML("<h3>Section 7 — Recent Auctions (All Metrics)</h3>"),
    _btn7, _out7,
]))


In [ ]:
# ─── Section 8 · Export to CSV ───────────────────────────────────────────────

def export_csv():
    df       = state["df"]
    upcoming = state["upcoming"]

    if df.empty:
        print("⚠️  No data to export — click 🔄 Refresh Data first."); return

    # Export directory next to the notebook
    notebook_dir = os.path.dirname(os.path.abspath("treasury_issuance_tracker.ipynb"))
    export_dir   = os.path.join(notebook_dir, "exports")
    os.makedirs(export_dir, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    # ── Historical auctions ──
    hist_path = os.path.join(export_dir, f"treasury_issuances_{ts}.csv")
    df_export = df.copy()
    for col in df_export.select_dtypes(include=["datetime64[ns]",
                                                 "datetime64[ns, UTC]"]).columns:
        df_export[col] = df_export[col].dt.strftime("%Y-%m-%d")
    df_export.to_csv(hist_path, index=False)
    print(f"✅ Historical auctions  →  {hist_path}")
    print(f"   {len(df_export):,} rows  ×  {len(df_export.columns)} columns")

    # ── Upcoming auctions ──
    if isinstance(upcoming, pd.DataFrame) and not upcoming.empty:
        up_path = os.path.join(export_dir, f"treasury_upcoming_{ts}.csv")
        df_up = upcoming.copy()
        for col in df_up.select_dtypes(include=["datetime64[ns]"]).columns:
            df_up[col] = df_up[col].dt.strftime("%Y-%m-%d")
        df_up.to_csv(up_path, index=False)
        print(f"✅ Upcoming auctions    →  {up_path}")
        print(f"   {len(df_up):,} rows  ×  {len(df_up.columns)} columns")

    print(f"\n📁 Export folder: {export_dir}")


_out8 = widgets.Output()
_btn8 = widgets.Button(description="💾 Export to CSV", button_style="success",
                       layout=widgets.Layout(width="180px"))
def _click8(_):
    with _out8: clear_output(wait=True); export_csv()
_btn8.on_click(_click8)
display(widgets.VBox([
    widgets.HTML(
        "<h3>Section 8 — Export Data</h3>"
        "<p>Saves CSVs to an <code>exports/</code> folder next to this notebook.</p>"
    ),
    _btn8, _out8,
]))


In [ ]:
# ─── Section 9 · Summary Dashboard ──────────────────────────────────────────

def show_summary():
    df = state["df"]
    if df.empty:
        print("⚠️  No data — click 🔄 Refresh Data first."); return

    agg = {
        "auctions":      ("auctionDate", "count"),
        "latest_auction":("auctionDate", "max"),
    }
    if "offeringAmount"    in df.columns: agg["total_offered_bn"]  = ("offeringAmount",    lambda x: x.sum()/1e9)
    if "bidToCoverRatio"   in df.columns: agg["avg_btc"]           = ("bidToCoverRatio",   "mean")
    if "highYield"         in df.columns: agg["latest_high_yield"] = ("highYield",         "last")
    if "totalTendered"     in df.columns: agg["total_tendered_bn"] = ("totalTendered",     lambda x: x.sum()/1e9)
    if "totalAccepted"     in df.columns: agg["total_accepted_bn"] = ("totalAccepted",     lambda x: x.sum()/1e9)

    summary = df.groupby("securityType").agg(**agg).reset_index()
    summary["latest_auction"] = summary["latest_auction"].dt.strftime("%Y-%m-%d")

    rename = {
        "securityType":     "Type",
        "auctions":         "Total Auctions",
        "total_offered_bn": "Total Offered (USD Bn)",
        "total_tendered_bn":"Total Tendered (USD Bn)",
        "total_accepted_bn":"Total Accepted (USD Bn)",
        "avg_btc":          "Avg Bid-to-Cover",
        "latest_high_yield":"Latest High Yield (%)",
        "latest_auction":   "Latest Auction",
    }
    summary = summary.rename(columns={k: v for k, v in rename.items() if k in summary.columns})

    fmt = {}
    if "Total Auctions"         in summary.columns: fmt["Total Auctions"]         = "{:,}"
    if "Total Offered (USD Bn)" in summary.columns: fmt["Total Offered (USD Bn)"] = "${:,.0f}B"
    if "Total Tendered (USD Bn)"in summary.columns: fmt["Total Tendered (USD Bn)"]= "${:,.0f}B"
    if "Total Accepted (USD Bn)"in summary.columns: fmt["Total Accepted (USD Bn)"]= "${:,.0f}B"
    if "Avg Bid-to-Cover"       in summary.columns: fmt["Avg Bid-to-Cover"]       = "{:.2f}×"
    if "Latest High Yield (%)"  in summary.columns: fmt["Latest High Yield (%)"]  = "{:.3f}%"

    styled = (
        summary.style
        .format(fmt, na_rep="—")
        .set_table_styles([
            {"selector":"th","props":[
                ("background-color","#1e3a5f"),("color","white"),
                ("font-weight","bold"),("padding","7px 16px"),
            ]},
            {"selector":"td","props":[
                ("padding","6px 16px"),("border","1px solid #ddd"),
                ("font-size","13px"),
            ]},
            {"selector":"tr:hover","props":[("background-color","#fffde7")]},
        ])
    )
    display(HTML("<h4>📋 All-Time Summary by Security Type</h4>"))
    display(styled)

    # Mini sparkline — latest 12 months offering amount
    if "offeringAmount" in df.columns and "auctionDate" in df.columns:
        cutoff = df["auctionDate"].max() - pd.DateOffset(months=12)
        df_spark = df[df["auctionDate"] >= cutoff].copy()
        df_spark["month"] = df_spark["auctionDate"].dt.to_period("M").dt.to_timestamp()
        spark = (
            df_spark.groupby(["month","securityType"])["offeringAmount"]
            .sum().reset_index()
        )
        spark["bn"] = spark["offeringAmount"] / 1e9
        fig = px.bar(
            spark, x="month", y="bn", color="securityType",
            color_discrete_map=TYPE_COLORS, barmode="stack",
            labels={"month":"Month","bn":"Offering (USD Bn)","securityType":"Type"},
            title="📅 Last 12 Months — Monthly Offering by Type",
        )
        fig.update_layout(height=380, template="plotly_white", hovermode="x unified")
        fig.show()


_out9 = widgets.Output()
_btn9 = widgets.Button(description="📋 Summary Dashboard", button_style="info",
                       layout=widgets.Layout(width="200px"))
def _click9(_):
    with _out9: clear_output(wait=True); show_summary()
_btn9.on_click(_click9)
display(widgets.VBox([widgets.HTML("<h3>Section 9 — Summary Dashboard</h3>"), _btn9, _out9]))


---
## 📚 Data Sources & Reference

| Resource | URL |
|----------|-----|
| TreasuryDirect Auction Search | `https://www.treasurydirect.gov/TA_WS/securities/search` |
| TreasuryDirect Upcoming Auctions | `https://www.treasurydirect.gov/TA_WS/securities/upcoming` |
| TreasuryDirect Website | https://www.treasurydirect.gov |

### Security Types Covered
| Type | Description |
|------|-------------|
| **Bill** | Short-term T-Bills (4, 8, 13, 17, 26, 52-week) |
| **Note** | Medium-term T-Notes (2, 3, 5, 7, 10-year) |
| **Bond** | Long-term T-Bonds (20, 30-year) |
| **CMB** | Cash Management Bills (flexible, short-term) |
| **TIPS** | Treasury Inflation-Protected Securities |
| **FRN** | Floating Rate Notes (2-year, indexed to SOFR) |

### Key Metrics Captured
- **Auction details**: CUSIP, security term, auction/issue/maturity dates, reopening status  
- **Issuance size**: Offering amount, total tendered, total accepted, bid-to-cover ratio  
- **Pricing**: High/low/median yield, investment rate, discount rate, price per $100, spread  
- **Bidder breakdown**: Primary dealers, direct bidders, indirect bidders, SOMA, FIMA, TreasuryDirect  
- **TIPS-specific**: CPI reference, index ratio  
- **FRN-specific**: Spread, FRN index determination rate  

### Export
CSV files are saved to `./exports/` folder next to this notebook.
